In [ ]:
##  Airline Flight Delay Analysis

In [ ]:
# ==============================
# Import Required Libraries
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_squared_error,
    r2_score,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_curve,
    auc
)


In [ ]:
# ============================================
#  Load Dataset
# ============================================

df = pd.read_csv('data/flights_data_100k.csv')

# Display first 5 rows
df.head()

In [ ]:
# ============================================
#  Data Inspection
# ============================================

print("Shape of dataset:", df.shape)
print("\nColumn Names:\n", df.columns.tolist())

# Check missing values
print("\nMissing Values:\n")
print(df.isnull().sum())

In [ ]:
# ============================================
#  Data Cleaning
# ============================================

# Columns to fill missing values with 0
cols_to_fill = [
    "DEP_TIME", "DEP_DELAY", "TAXI_OUT", "WHEELS_OFF",
    "WHEELS_ON", "TAXI_IN", "ARR_TIME", "ARR_DELAY",
    "CANCELLATION_CODE", "CRS_ELAPSED_TIME", "ELAPSED_TIME",
    "AIR_TIME", "DELAY_DUE_CARRIER", "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS", "DELAY_DUE_SECURITY", "DELAY_DUE_LATE_AIRCRAFT"
]

df[cols_to_fill] = df[cols_to_fill].fillna(0)

# Save cleaned dataset
df.to_csv("data/Cleaned_flights_data.csv", index=False)

print("✅ Missing values handled and cleaned dataset saved.")

In [ ]:
## Exploratory Data Analysis

In [ ]:
# Arrival Delay Distribution

plt.figure(figsize=(8,5))
sns.histplot(df['ARR_DELAY'], bins=50, kde=True)
plt.title("Distribution of Arrival Delay")
plt.xlabel("Arrival Delay (minutes)")
plt.ylabel("Number of Flights")
plt.show()

In [ ]:
# Flights per Airline

plt.figure(figsize=(10,5))
sns.countplot(x='AIRLINE', data=df)
plt.title("Number of Flights per Airline")
plt.xticks(rotation=45)
plt.show()

In [ ]:
# Cancellation Analysis

sns.countplot(x='CANCELLED', data=df)
plt.title("Cancelled vs Non-Cancelled Flights")
plt.show()

In [ ]:
# Delay Contribution by Reason

delay_cols = [
    'DELAY_DUE_CARRIER',
    'DELAY_DUE_WEATHER',
    'DELAY_DUE_NAS',
    'DELAY_DUE_SECURITY',
    'DELAY_DUE_LATE_AIRCRAFT'
]

delay_sum = df[delay_cols].sum().reset_index()
delay_sum.columns = ['Delay_Type', 'Total_Delay']

plt.figure(figsize=(10,5))
sns.barplot(x='Delay_Type', y='Total_Delay', data=delay_sum)
plt.xticks(rotation=45)
plt.title("Total Delay Contribution by Type")
plt.show()

In [ ]:
# Correlation Heatmap

numeric_cols = [
    'DEP_DELAY','ARR_DELAY','TAXI_OUT','TAXI_IN',
    'AIR_TIME','DISTANCE','ELAPSED_TIME'
]

plt.figure(figsize=(8,6))
sns.heatmap(df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()


In [ ]:
## Linear Regression (Predict Arrival Delay)

In [ ]:
# Features
X = df[['DEP_DELAY', 'DISTANCE', 'AIR_TIME', 'TAXI_OUT']]
y = df['ARR_DELAY']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Model
lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

# Predictions
y_pred = lr_model.predict(X_test)

# Evaluation
print("Mean Squared Error:", mean_squared_error(y_test, y_pred))
print("R2 Score:", r2_score(y_test, y_pred))

In [ ]:
## Logistic Regression (Classify Delayed Flights)

In [ ]:
# Create binary target (Delayed if ARR_DELAY > 15 mins)
df['DELAYED'] = (df['ARR_DELAY'] > 15).astype(int)

X = df[['DEP_DELAY', 'DISTANCE', 'AIR_TIME', 'TAXI_OUT']]
y = df['DELAYED']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

log_model = LogisticRegression()
log_model.fit(X_train, y_train)

y_pred = log_model.predict(X_test)
y_prob = log_model.predict_proba(X_test)[:,1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))

In [ ]:
## MySQL Database Connection

In [ ]:
# ============================================
# Connect to MySQL Database
# ============================================

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="airline"
)

if db.is_connected():
    print("✅ Connected to MySQL successfully")

db.close()

In [ ]:
# Insert Cleaned Data into MySQL

import csv

def to_bool(value):
    return 1 if str(value).strip().lower() in ('1', 'true', 'yes') else 0

db = mysql.connector.connect(
    host="localhost",
    user="root",
    password="your_password",
    database="airline"
)

cursor = db.cursor()

sql = """
INSERT INTO flights (
    FL_DATE, AIRLINE, AIRLINE_DOT, AIRLINE_CODE, DOT_CODE, FL_NUMBER,
    ORIGIN, ORIGIN_CITY, DEST, DEST_CITY,
    CRS_DEP_TIME, DEP_TIME, DEP_DELAY,
    TAXI_OUT, WHEELS_OFF, WHEELS_ON, TAXI_IN,
    CRS_ARR_TIME, ARR_TIME, ARR_DELAY,
    CANCELLED, CANCELLATION_CODE, DIVERTED,
    CRS_ELAPSED_TIME, ELAPSED_TIME, AIR_TIME, DISTANCE,
    DELAY_DUE_CARRIER, DELAY_DUE_WEATHER, DELAY_DUE_NAS,
    DELAY_DUE_SECURITY, DELAY_DUE_LATE_AIRCRAFT
)
VALUES (%s, %s, %s, %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s,
        %s, %s, %s, %s,
        %s, %s, %s,
        %s, %s)
"""

with open("data/Cleaned_flights_data.csv", newline="", encoding="utf-8") as file:
    reader = csv.reader(file)
    next(reader)

    for row in reader:
        row[20] = to_bool(row[20])
        row[22] = to_bool(row[22])
        cursor.execute(sql, row)

db.commit()
cursor.close()
db.close()

print("✅ Data inserted successfully into MySQL.")

In [ ]:
## Key Business Insights

- Departure delays strongly influence arrival delays across flights.
- Weather and late aircraft are major contributors to total delay minutes.
- Taxi-out time and operational congestion increase delay probability.
- Delay patterns vary across airlines, indicating performance differences.
- Logistic Regression effectively predicts flights delayed by more than 15 minutes.